In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

In [3]:
# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv("retail_sales_data.csv", parse_dates=["OrderDate"])

print("=" * 60)
print("STEP 1: RAW DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(df.head())
print("\nInfo:")
print(df.info())


STEP 1: RAW DATA OVERVIEW
Shape: (1210, 14)
   OrderID  OrderDate CustomerID CustomerSegment   Region ProductCategory  \
0  ORD1000 2024-04-12    CUST385     Home Office    North       Groceries   
1  ORD1001 2025-03-11    CUST243        Consumer  Central        Clothing   
2  ORD1002 2024-09-27    CUST118       Corporate     West        Clothing   
3  ORD1003 2024-04-16    CUST117       Corporate    South        Clothing   
4  ORD1004 2024-03-12    CUST356       Corporate     West     Electronics   

       Product  Quantity  UnitPrice  DiscountPct      Sales   Profit  \
0  Cooking Oil         5     994.14         0.05    4722.16  1573.92   
1          Cap         3     751.98         0.20    1804.75  -110.55   
2     Sneakers         3    3246.01         0.25    7303.52  -661.99   
3     Sneakers         1     599.63          NaN     569.65   147.50   
4       Tablet         3   57393.66         0.15  146353.83  9022.90   

   ShippingCost       PaymentMode  
0        259.39         

In [4]:

# ---------------------------------------------------------
# 2. DATA CLEANING
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 2: DATA CLEANING")
print("=" * 60)

print("Missing values before cleaning:\n", df.isnull().sum())

# Fill missing discount with 0 (assume no discount applied)
df["DiscountPct"] = df["DiscountPct"].fillna(0)

# Remove duplicate orders (same OrderID)
before = len(df)
df = df.drop_duplicates(subset="OrderID")
print(f"\nRemoved {before - len(df)} duplicate rows.")

# Add useful derived columns
df["Month"] = df["OrderDate"].dt.to_period("M").astype(str)
df["ProfitMargin"] = (df["Profit"] / df["Sales"] * 100).round(2)

print("\nMissing values after cleaning:\n", df.isnull().sum())



STEP 2: DATA CLEANING
Missing values before cleaning:
 OrderID             0
OrderDate           0
CustomerID          0
CustomerSegment     0
Region              0
ProductCategory     0
Product             0
Quantity            0
UnitPrice           0
DiscountPct        25
Sales               0
Profit              0
ShippingCost        0
PaymentMode         0
dtype: int64

Removed 10 duplicate rows.

Missing values after cleaning:
 OrderID            0
OrderDate          0
CustomerID         0
CustomerSegment    0
Region             0
ProductCategory    0
Product            0
Quantity           0
UnitPrice          0
DiscountPct        0
Sales              0
Profit             0
ShippingCost       0
PaymentMode        0
Month              0
ProfitMargin       0
dtype: int64


In [5]:

# ---------------------------------------------------------
# 3. NUMPY-BASED NUMERICAL SUMMARY
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 3: NUMPY SUMMARY STATISTICS")
print("=" * 60)

sales_arr = df["Sales"].to_numpy()
profit_arr = df["Profit"].to_numpy()

print(f"Total Sales      : {np.sum(sales_arr):,.2f}")
print(f"Average Sales    : {np.mean(sales_arr):,.2f}")
print(f"Median Sales     : {np.median(sales_arr):,.2f}")
print(f"Std Dev (Sales)  : {np.std(sales_arr):,.2f}")
print(f"Total Profit     : {np.sum(profit_arr):,.2f}")
print(f"Loss-making orders (Profit < 0): {np.sum(profit_arr < 0)}")
print(f"95th percentile Sales value: {np.percentile(sales_arr, 95):,.2f}")



STEP 3: NUMPY SUMMARY STATISTICS
Total Sales      : 44,067,380.19
Average Sales    : 36,722.82
Median Sales     : 6,773.19
Std Dev (Sales)  : 62,947.11
Total Profit     : 11,030,657.90
Loss-making orders (Profit < 0): 43
95th percentile Sales value: 167,084.65


In [6]:
# ---------------------------------------------------------
# 4. PANDAS EDA / AGGREGATIONS
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 4: PANDAS GROUP-BY INSIGHTS")
print("=" * 60)

sales_by_category = df.groupby("ProductCategory")["Sales"].sum().sort_values(ascending=False)
print("\nSales by Category:\n", sales_by_category)

sales_by_region = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
print("\nSales by Region:\n", sales_by_region)

profit_by_segment = df.groupby("CustomerSegment")["Profit"].mean().sort_values(ascending=False)
print("\nAverage Profit by Customer Segment:\n", profit_by_segment)

monthly_sales = df.groupby("Month")["Sales"].sum()
print("\nMonthly Sales Trend (first 5 months):\n", monthly_sales.head())

top_products = df.groupby("Product")["Sales"].sum().sort_values(ascending=False).head(5)
print("\nTop 5 Products by Sales:\n", top_products)



STEP 4: PANDAS GROUP-BY INSIGHTS

Sales by Category:
 ProductCategory
Electronics    27333239.52
Furniture      14033632.59
Clothing        1499881.31
Groceries        663811.54
Stationery       536815.23
Name: Sales, dtype: float64

Sales by Region:
 Region
West       10576264.88
North       9988027.26
East        9070195.78
Central     7293144.37
South       7139747.90
Name: Sales, dtype: float64

Average Profit by Customer Segment:
 CustomerSegment
Corporate      10183.362668
Home Office    10180.810678
Consumer        8359.857147
Name: Profit, dtype: float64

Monthly Sales Trend (first 5 months):
 Month
2024-01    1805377.21
2024-02    2153519.33
2024-03    1408608.40
2024-04    2273292.83
2024-05    1254009.33
Name: Sales, dtype: float64

Top 5 Products by Sales:
 Product
Smartphone    8225295.75
Tablet        6009699.40
Headphones    5061934.83
Smartwatch    4835457.61
Laptop        3200851.93
Name: Sales, dtype: float64


In [7]:
# ---------------------------------------------------------
# 5. VISUALIZATIONS (Matplotlib + Seaborn)
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 5: GENERATING CHARTS (saved as PNG files)")
print("=" * 60)

# Chart 1: Sales by Category (Bar)
plt.figure()
sns.barplot(x=sales_by_category.values, y=sales_by_category.index, hue=sales_by_category.index,
            palette="viridis", legend=False)
plt.title("Total Sales by Product Category")
plt.xlabel("Total Sales")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("chart1_sales_by_category.png", dpi=150)
plt.close()


STEP 5: GENERATING CHARTS (saved as PNG files)


In [8]:
# Chart 2: Monthly Sales Trend (Line)
plt.figure()
monthly_sales.plot(kind="line", marker="o", color="teal")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("chart2_monthly_sales_trend.png", dpi=150)
plt.close()

In [9]:
# Chart 3: Sales Distribution (Histogram)
plt.figure()
sns.histplot(df["Sales"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Sales Values")
plt.xlabel("Sales Amount")
plt.tight_layout()
plt.savefig("chart3_sales_distribution.png", dpi=150)
plt.close()

In [10]:
# Chart 4: Region vs Category Sales (Heatmap)
pivot = df.pivot_table(values="Sales", index="Region", columns="ProductCategory", aggfunc="sum")
plt.figure(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Sales Heatmap: Region vs Category")
plt.tight_layout()
plt.savefig("chart4_region_category_heatmap.png", dpi=150)
plt.close()

In [11]:
# Chart 5: Profit Margin by Segment (Boxplot)
plt.figure()
sns.boxplot(data=df, x="CustomerSegment", y="ProfitMargin", hue="CustomerSegment",
            palette="Set2", legend=False)
plt.title("Profit Margin Spread by Customer Segment")
plt.tight_layout()
plt.savefig("chart5_profit_margin_boxplot.png", dpi=150)
plt.close()

In [12]:
# Chart 6: Discount vs Profit (Scatter)
plt.figure()
sns.scatterplot(data=df, x="DiscountPct", y="Profit", hue="ProductCategory", alpha=0.6)
plt.title("Discount % vs Profit")
plt.tight_layout()
plt.savefig("chart6_discount_vs_profit.png", dpi=150)
plt.close()

print("6 charts saved in the current folder.")

6 charts saved in the current folder.


In [13]:
# ---------------------------------------------------------
# 6. KEY INSIGHTS SUMMARY
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 6: KEY BUSINESS INSIGHTS")
print("=" * 60)
print(f"1. Best-selling category : {sales_by_category.idxmax()}")
print(f"2. Top region by sales   : {sales_by_region.idxmax()}")
print(f"3. Most profitable segment (avg): {profit_by_segment.idxmax()}")
print(f"4. Best-selling product  : {top_products.idxmax()}")
print(f"5. % of loss-making orders: {(profit_arr < 0).mean()*100:.2f}%")

print("\nDone. Review the printed insights and the 6 PNG charts generated.")



STEP 6: KEY BUSINESS INSIGHTS
1. Best-selling category : Electronics
2. Top region by sales   : West
3. Most profitable segment (avg): Corporate
4. Best-selling product  : Smartphone
5. % of loss-making orders: 3.58%

Done. Review the printed insights and the 6 PNG charts generated.
